# 4. Mathematisches Modell: Gefahrgut-Routenplanung & Fahrzeugzuweisung



 Ziel ist die Minimierung von **Transportrisiko** (Bevölkerungsdichte, Naturschutzgebiete, Unfallwahrscheinlichkeit) und **Transportkosten** (Fixkosten, variable Energie- und Kilometerkosten).

---

## Inhaltsverzeichnis
1. [Abhängigkeiten & Imports](#imports)
2. [Sets (Mengen)](#sets)
3. [Parameter](#parameter)
4. [Entscheidungsvariablen](#variables)
5. [Zielfunktion](#objective)
6. [Nebenbedingungen](#constraints)
7. [Modell lösen & Ergebnisse](#solve)

---
## 1. Abhängigkeiten & Imports <a id='imports'></a>

In [1]:
# Installiere PuLP falls noch nicht vorhanden
# !pip install pulp

import pulp as pl
import pandas as pd
import numpy as np
import itertools
import warnings
warnings.filterwarnings('ignore')

print(f"PuLP Version: {pl.__version__}")
print(f"Verfügbare Solver: {pl.listSolvers(onlyAvailable=True)}")

PuLP Version: 3.3.0
Verfügbare Solver: ['PULP_CBC_CMD']


---
## 2. Sets (Mengen) <a id='sets'></a>

| Symbol | Bedeutung |
|--------|----------|
| $V$ | Menge der verfügbaren Fahrzeuge |
| $L$ | Menge der durchzuführenden Gefahrgutlieferungen |
| $N$ | Menge aller Knoten (Kreuzungen, Depots, Ziele) im Netz |
| $E$ | Menge aller gerichteten Kanten (Straßenabschnitte) |
| $K$ | Menge der Gefahrgutklassen |

> **Hinweis:** Männer die Dummy-Werte unten dienen nur als Testfall. Für die reale Anwendung müssen wir noch $N$ und $E$ aus einem Straßengraphen (OSM / eigene Daten) und $L$ aus den Auftragsdaten befüllt werden. (Entweder wir lassen uns diese Werte mit KI generieren oder fidnen selbst welche)

In [ ]:
# ---------------------------------------------------------------------------
# SETS — Dummy-Instanz 
# ---------------------------------------------------------------------------

# Fahrzeuge (aus vehicles.csv)
V = ['MAN_eTGX', 'Volvo_FH_Electric', 'Mercedes_eActros_600']

# Gefahrgutklassen (ADR-Klassen)
K = ['klasse_3',   # Entzündbare Flüssigkeiten
     'klasse_6',   # Giftige Stoffe
     'klasse_8']   # Ätzende Stoffe

# Knoten im Straßennetz: 0 = Depot, 1-3 = Zwischenknoten, 4-5 = Ziele
N = ['depot', 'n1', 'n2', 'n3', 'ziel_A', 'ziel_B']

# Gerichtete Kanten (i -> j)
E = [
    ('depot', 'n1'), ('depot', 'n2'),
    ('n1',  'n2'),   ('n1',  'n3'),   ('n1',  'ziel_A'),
    ('n2',  'n1'),   ('n2',  'n3'),   ('n2',  'ziel_B'),
    ('n3',  'ziel_A'), ('n3', 'ziel_B'),
    ('ziel_A', 'depot'), ('ziel_B', 'depot')  # Rückfahrt möglich
]

# Lieferungen
L = ['lief_1', 'lief_2', 'lief_3']

print(f"Fahrzeuge (|V|={len(V)}): {V}")
print(f"Gefahrgutklassen (|K|={len(K)}): {K}")
print(f"Knoten (|N|={len(N)}): {N}")
print(f"Kanten (|E|={len(E)}): {len(E)} gerichtete Kanten")
print(f"Lieferungen (|L|={len(L)}): {L}")
print()
print(f"Variablen x[l,v,e] (naive Formulierung): {len(L)*len(V)*len(E):,}")
print(f"Variablen f[l,e] (entkoppelte Formulierung): {len(L)*len(E):,}  ← empfohlen")

Fahrzeuge (|V|=3): ['MAN_eTGX', 'Volvo_FH_Electric', 'Mercedes_eActros_600']
Gefahrgutklassen (|K|=3): ['klasse_3', 'klasse_6', 'klasse_8']
Knoten (|N|=6): ['depot', 'n1', 'n2', 'n3', 'ziel_A', 'ziel_B']
Kanten (|E|=12): 12 gerichtete Kanten
Lieferungen (|L|=3): ['lief_1', 'lief_2', 'lief_3']

Variablen x[l,v,e] (naive Formulierung): 108
Variablen f[l,e] (entkoppelte Formulierung): 36  ← empfohlen


---
## 3. Parameter <a id='parameter'></a>

| Symbol | Einheit | Bedeutung |
|--------|---------|----------|
| $Cap_v$ | t | Maximale Nutzlast von Fahrzeug $v$ |
| $FC_v$ | €/Tag | Fixkosten für den Einsatz von $v$ |
| $VC_{v,e}$ | € | Variable Kosten für $v$ auf Kante $e$ (Energie + km) |
| $Dem_l$ | t | Gewicht der Lieferung $l$ |
| $Class_l$ | — | Gefahrgutklasse $k \in K$ der Lieferung $l$ |
| $O_l, D_l$ | — | Start- und Zielknoten der Lieferung $l$ |
| $Risk_{e,k}$ | — | Risikoscore für Kante $e$ und Klasse $k$ (aggregiert aus Daten) |
| $Allow_{e,k}$ | $\{0,1\}$ | 1 wenn Kante $e$ für Klasse $k$ rechtlich freigegeben |
| $Dist_e$ | km | Länge der Kante $e$ |

### 3.1 Fahrzeugparameter
> Quelle: habe das aus `vehicles.csv` genommen

In [3]:
# ---------------------------------------------------------------------------
# FAHRZEUGPARAMETER — aus vehicles.csv
# ---------------------------------------------------------------------------

vehicles_data = {
    'MAN_eTGX':             {'Cap': 18, 'FC': 180, 'VC_per_km': 0.55,
                              'battery_kwh': 480, 'range_km': 500,
                              'energy_kwh_per_km': 0.96, 'charging_kw': 375},
    'Volvo_FH_Electric':    {'Cap': 20, 'FC': 200, 'VC_per_km': 0.60,
                              'battery_kwh': 540, 'range_km': 470,
                              'energy_kwh_per_km': 1.15, 'charging_kw': 350},
    'Mercedes_eActros_600': {'Cap': 22, 'FC': 220, 'VC_per_km': 0.65,
                              'battery_kwh': 621, 'range_km': 500,
                              'energy_kwh_per_km': 1.20, 'charging_kw': 400},
}

# Energie-Szenarien (aus energy_prices.csv)
energy_prices = {
    'depot':       0.35,  # €/kWh — Laden am Depot
    'urban_ac':    0.55,  # €/kWh — Stadtladen AC
    'urban_dc':    0.60,  # €/kWh — Stadtladen DC
    'highway_hpc': 0.75,  # €/kWh — Autobahn HPC
}

# Hilfsdictionaries für den Solver
Cap = {v: vehicles_data[v]['Cap']       for v in V}
FC  = {v: vehicles_data[v]['FC']        for v in V}

pd.DataFrame(vehicles_data).T.style.format(precision=2)

,Cap,FC,VC_per_km,battery_kwh,range_km,energy_kwh_per_km,charging_kw
MAN_eTGX,18.00,180.00,0.55,480.00,500.00,0.96,375.00
Volvo_FH_Electric,20.00,200.00,0.60,540.00,470.00,1.15,350.00
Mercedes_eActros_600,22.00,220.00,0.65,621.00,500.00,1.20,400.00


### 3.2 Netzwerk- & Risikoparameter

Der **Risikoscore** $Risk_{e,k}$ aggregiert drei Datenquellen:

$$Risk_{e,k} = \underbrace{\alpha \cdot \text{PopDens}_e}_{\text{Bevölkerungsdichte}} + \underbrace{\beta \cdot \text{AccRate}_e}_{\text{Unfallrate}} + \underbrace{\gamma \cdot \text{NatRes}_e}_{\text{Naturschutz}} \quad \text{mit } \alpha + \beta + \gamma = 1$$

- $\text{PopDens}_e$: aus `density.csv` (JRC Census Raster, 100m Auflösung)
- $\text{AccRate}_e$: aus `Accidents.csv` (268k Unfallereignisse → Spatial Join auf Kanten)
- $\text{NatRes}_e$: aus `nature_reserves_filtered.csv` (Naturschutzgebiete NW/NI)

> **TODO Preprocessing:** Spatial Join der Punkt-Daten auf Kanten noch nicht implementiert.
> Platzhalter-Werte werden unten verwendet.

In [ ]:
# ---------------------------------------------------------------------------
# NETZWERKPARAMETER — Platzhalter (das müssen wir durch Spatial-Join-Pipeline ersetzen später)
# ---------------------------------------------------------------------------

# Kantenlängen in km (Dummy)
Dist = {
    ('depot','n1'): 45, ('depot','n2'): 60,
    ('n1','n2'):    20, ('n1','n3'):    35, ('n1','ziel_A'): 50,
    ('n2','n1'):    20, ('n2','n3'):    25, ('n2','ziel_B'): 40,
    ('n3','ziel_A'):30, ('n3','ziel_B'): 15,
    ('ziel_A','depot'):80, ('ziel_B','depot'): 70,
}

# Risikoscores pro Kante und Gefahrgutklasse [0, 1] normiert
# Quelle: aggregiert aus PopDens + AccRate + NatRes (TODO: echte Werte)
np.random.seed(42)  # Reproduzierbarkeit
Risk = {
    (e, k): round(np.random.uniform(0.1, 0.9), 3)
    for e in E for k in K
}

# Erlaubnismatrix: Allow[e][k] = 1 wenn Kante e für Klasse k freigegeben
# Hauptstraßen erlaubt für alle, Naturschutzgebiet-nahe Kanten gesperrt
Allow = {(e, k): 1 for e in E for k in K}
# Beispiel: n3 -> ziel_A liegt im Naturschutzgebiet → gesperrt für Klasse 6+8
Allow[('n3', 'ziel_A'), 'klasse_6'] = 0
Allow[('n3', 'ziel_A'), 'klasse_8'] = 0

# Variable Kosten: VC[v][e] = Distanz * VC_per_km + Energie * Energiepreis
# Energiepreis abhängig vom Kanten-Typ (highway → HPC, urban → DC)
edge_type = {e: 'highway_hpc' if Dist[e] > 40 else 'urban_dc' for e in E}

VC = {}
for v in V:
    for e in E:
        km_cost     = Dist[e] * vehicles_data[v]['VC_per_km']
        energy_cost = Dist[e] * vehicles_data[v]['energy_kwh_per_km'] * energy_prices[edge_type[e]]
        VC[(v, e)]  = round(km_cost + energy_cost, 2)

print("Variable Kosten VC[v][e] — Auszug:")
sample_edges = list(E)[:4]
vc_table = pd.DataFrame(
    {e: {v: VC[(v, e)] for v in V} for e in sample_edges}
).T
vc_table.columns.name = 'Fahrzeug'
vc_table.index.name   = 'Kante'
print(vc_table.to_string())

Variable Kosten VC[v][e] — Auszug:
Fahrzeug  MAN_eTGX  Volvo_FH_Electric  Mercedes_eActros_600
depot n1     57.15              65.81                 69.75
      n2     76.20              87.75                 93.00
n1    n2     22.52              25.80                 27.40
      n3     39.41              45.15                 47.95


### 3.3 Lieferungsparameter

In [ ]:
# ---------------------------------------------------------------------------
# LIEFERUNGSPARAMETER
# ---------------------------------------------------------------------------

deliveries = {
    'lief_1': {'Dem': 15, 'Class': 'klasse_3', 'O': 'depot', 'D': 'ziel_A'},
    'lief_2': {'Dem':  8, 'Class': 'klasse_6', 'O': 'depot', 'D': 'ziel_B'},
    'lief_3': {'Dem': 12, 'Class': 'klasse_8', 'O': 'depot', 'D': 'ziel_A'},
}

Dem   = {l: deliveries[l]['Dem']   for l in L}
Class = {l: deliveries[l]['Class'] for l in L}
O_dep = {l: deliveries[l]['O']     for l in L}  # Origin
D_dep = {l: deliveries[l]['D']     for l in L}  # Destination

print("Lieferungsparameter:")
pd.DataFrame(deliveries).T

---
## 4. Entscheidungsvariablen <a id='variables'></a>

Wir verwenden eine **entkoppelte Formulierung**, die die Variablenanzahl drastisch reduziert:

| Variable | Typ | Bedeutung |
|----------|-----|-----------|
| $f_{l,e} \in \{0,1\}$ | Binär | 1, wenn Lieferung $l$ über Kante $e$ transportiert wird |
| $y_{l,v} \in \{0,1\}$ | Binär | 1, wenn Lieferung $l$ dem Fahrzeug $v$ zugewiesen wird |
| $z_v \in \{0,1\}$ | Binär | 1, wenn Fahrzeug $v$ heute eingesetzt wird |
| $u_{l,n} \geq 0$ | Stetig | MTZ-Hilfsvariable (Besuchsreihenfolge an Knoten $n$ für Lieferung $l$) |

> **Warum entkoppelt?** Die naive Formulierung $x_{l,v,e}$ erzeugt $|L| \times |V| \times |E|$ Variablen.  
> Die entkoppelte Variante $f_{l,e}$ benötigt nur $|L| \times |E|$ — ein Faktor $|V|$ weniger.

In [5]:
# ---------------------------------------------------------------------------
# MODELL INITIALISIERUNG
# ---------------------------------------------------------------------------

model = pl.LpProblem("Gefahrgut_Routenplanung_MILP", pl.LpMinimize)

# ---------------------------------------------------------------------------
# ENTSCHEIDUNGSVARIABLEN
# ---------------------------------------------------------------------------

# f[l, e]: Flussvariable — Lieferung l auf Kante e (fahrzeugagnostisch)
f = pl.LpVariable.dicts(
    "Fluss",
    [(l, e) for l in L for e in E],
    cat='Binary'
)

# y[l, v]: Zuweisungsvariable — Lieferung l an Fahrzeug v
y = pl.LpVariable.dicts(
    "Zuweisung",
    [(l, v) for l in L for v in V],
    cat='Binary'
)

# z[v]: Aktivierungsvariable — Fahrzeug v wird eingesetzt
z = pl.LpVariable.dicts(
    "Aktivierung",
    [v for v in V],
    cat='Binary'
)

# u[l, n]: MTZ-Hilfsvariable für Subtour-Eliminierung
u = pl.LpVariable.dicts(
    "MTZ",
    [(l, n) for l in L for n in N],
    lowBound=0,
    upBound=len(N),
    cat='Continuous'
)

n_vars = len(f) + len(y) + len(z) + len(u)
print(f"Variablenanzahl gesamt: {n_vars}")
print(f"  f[l,e]  (Fluss):       {len(f):>6}  binär")
print(f"  y[l,v]  (Zuweisung):   {len(y):>6}  binär")
print(f"  z[v]    (Aktivierung): {len(z):>6}  binär")
print(f"  u[l,n]  (MTZ):         {len(u):>6}  stetig")

Variablenanzahl gesamt: 66
  f[l,e]  (Fluss):           36  binär
  y[l,v]  (Zuweisung):        9  binär
  z[v]    (Aktivierung):      3  binär
  u[l,n]  (MTZ):             18  stetig


---
## 5. Zielfunktion <a id='objective'></a>

Wir minimieren eine **normierte gewichtete Summe** aus Risiko und Kosten:

$$\min \quad \frac{w_1}{Risk_{\max}} \sum_{l \in L} \sum_{e \in E} Risk_{e, Class_l} \cdot f_{l,e} \quad + \quad \frac{w_2}{Cost_{\max}} \left( \sum_{v \in V} FC_v \cdot z_v \;+\; \sum_{l \in L} \sum_{v \in V} \sum_{e \in E} VC_{v,e} \cdot y_{l,v} \cdot \frac{f_{l,e}}{1} \right)$$

> **Normierung:** Beide Terme werden durch ihre jeweiligen Maximalwerte dividiert, um Einheiteninkonsistenz zu vermeiden. Mit $w_1 = w_2 = 0.5$ werden Risiko und Kosten gleichgewichtet.

In [6]:
# ---------------------------------------------------------------------------
# ZIELFUNKTION
# ---------------------------------------------------------------------------

# Gewichtungsparameter
w1 = 0.5  # Risiko-Gewicht
w2 = 0.5  # Kosten-Gewicht

# Normierungsschranken
Risk_max = max(Risk[(e, k)] for e in E for k in K)  # theoretisches Maximum
Risk_sum_max = Risk_max * len(L) * len(E)            # obere Schranke Risiko-Term

Cost_max = (sum(FC[v] for v in V)                    # alle Fahrzeuge aktiv
          + sum(max(VC[(v, e)] for v in V)            # teuerste Variante je Kante
                for e in E) * len(L))                  # für alle Lieferungen

print(f"Normierungsschranken:")
print(f"  Risk_sum_max = {Risk_sum_max:.3f}")
print(f"  Cost_max     = {Cost_max:.2f} €")
print()

# --- Risiko-Term ---
# Summe der Risikoscores über alle Kanten, gewichtet nach Gefahrgutklasse der Lieferung
risk_term = pl.lpSum(
    Risk[(e, Class[l])] * f[(l, e)]
    for l in L
    for e in E
)

# --- Kosten-Term ---
# Fixkosten aktivierter Fahrzeuge + variable Kosten
# Näherung: VC-Kosten werden pro Lieferung über die zugewiesene Fahrzeugklasse bestimmt
# (Linearisierung: da y[l,v] * f[l,e] bilinear wäre, nutzen wir die Zuweisung
#  zur Berechnung der erwarteten Kosten als zusätzlichen linearen Term)

fixed_cost_term = pl.lpSum(FC[v] * z[v] for v in V)

variable_cost_term = pl.lpSum(
    VC[(v, e)] * y[(l, v)] * (1 / len(V))  # Gleichverteilungsannahme über Kanten
    for l in L for v in V for e in E
    if Allow.get(((e), Class[l]), 1) == 1
)

# Normierte Zielfunktion
model += (
    w1 * risk_term / Risk_sum_max
    + w2 * (fixed_cost_term + variable_cost_term) / Cost_max
), "Zielfunktion_normiert"

print("Zielfunktion erfolgreich hinzugefügt.")

Normierungsschranken:
  Risk_sum_max = 31.536
  Cost_max     = 2778.60 €



NameError: name 'Class' is not defined

---
## 6. Nebenbedingungen <a id='constraints'></a>

### Übersicht aller Constraints

| # | Name | Formel | Zweck |
|---|------|--------|-------|
| C1 | Transportpflicht | $\sum_v y_{l,v} = 1$ | Jede Lieferung genau einem Fahrzeug |
| C2 | Kapazität | $\sum_l Dem_l \cdot y_{l,v} \leq Cap_v \cdot z_v$ | Nutzlastgrenze je Fahrzeug |
| C3 | Aktivierung | $z_v \geq y_{l,v}$ | Fahrzeug muss aktiv sein wenn Lieferung zugewiesen |
| C4 | Gefahrgutrestriktion | $f_{l,e} \leq Allow_{e,Class_l}$ | Legale Routenführung |
| C5 | Flusserhaltung | Netzwerkfluss-Gleichung | Physikalisch konsistente Route |
| C6 | Subtour-Eliminierung | MTZ-Ungleichung | Keine isolierten Schleifen |
| C7 | Reichweite | $\sum_e Dist_e \cdot f_{l,e} \leq Range_v \cdot y_{l,v}$ | Akkureichweite nicht überschreiten |

In [7]:
# ---------------------------------------------------------------------------
# C1: TRANSPORTPFLICHT
# Jede Lieferung muss exakt einem Fahrzeug zugewiesen werden.
#
#   sum_{v in V} y[l,v] = 1    für alle l in L
# ---------------------------------------------------------------------------

for l in L:
    model += (
        pl.lpSum(y[(l, v)] for v in V) == 1,
        f"C1_Transportpflicht_{l}"
    )

print(f"C1: {len(L)} Constraints hinzugefügt (Transportpflicht).")

C1: 3 Constraints hinzugefügt (Transportpflicht).


In [8]:
# ---------------------------------------------------------------------------
# C2: KAPAZITÄTSBEDINGUNG
# Die Gesamtlast zugewiesener Lieferungen darf die Nutzlast des Fahrzeugs
# nicht überschreiten. Ist z[v]=0, erzwingt die rechte Seite = 0.
#
#   sum_{l in L} Dem[l] * y[l,v]  <=  Cap[v] * z[v]    für alle v in V
# ---------------------------------------------------------------------------

for v in V:
    model += (
        pl.lpSum(Dem[l] * y[(l, v)] for l in L) <= Cap[v] * z[v],
        f"C2_Kapazitaet_{v}"
    )

print(f"C2: {len(V)} Constraints hinzugefügt (Kapazität).")

NameError: name 'Dem' is not defined

In [9]:
# ---------------------------------------------------------------------------
# C3: AKTIVIERUNGSBEDINGUNG
# Wird Fahrzeug v eine Lieferung zugewiesen, muss z[v] = 1 sein.
# Ohne diesen Constraint könnte C2 verletzt werden (z[v]=0, y[l,v]=1).
#
#   z[v] >= y[l,v]    für alle l in L, v in V
# ---------------------------------------------------------------------------

n_c3 = 0
for l in L:
    for v in V:
        model += (
            z[v] >= y[(l, v)],
            f"C3_Aktivierung_{l}_{v}"
        )
        n_c3 += 1

print(f"C3: {n_c3} Constraints hinzugefügt (Aktivierung).")

C3: 9 Constraints hinzugefügt (Aktivierung).


In [ ]:
# ---------------------------------------------------------------------------
# C4: GEFAHRGUTRESTRIKTIONEN
# Eine Kante darf nur befahren werden, wenn sie für die Gefahrgutklasse
# der Lieferung rechtlich freigegeben ist (Allow[e][k] = 1).
#
#   f[l,e] <= Allow[e, Class[l]]    für alle l in L, e in E
# ---------------------------------------------------------------------------

n_c4 = 0
for l in L:
    for e in E:
        permission = Allow.get((e, Class[l]), 1)  # Default: erlaubt
        if permission == 0:
            # Kante explizit gesperrt → f[l,e] muss 0 sein
            model += (
                f[(l, e)] <= 0,
                f"C4_Gefahrgut_{l}_{e[0]}_{e[1]}"
            )
            n_c4 += 1

print(f"C4: {n_c4} aktive Sperr-Constraints hinzugefügt (Gefahrgutrestriktionen).")
print("     (Nur gesperrte Kanten erhalten einen Constraint — effizienter als alle.)")

In [ ]:
# ---------------------------------------------------------------------------
# C5: FLUSSERHALTUNG (Network Flow Conservation)
# An jedem Knoten n gilt: eingehender Fluss = ausgehender Fluss,
# außer am Startknoten O[l] (netto -1) und Zielknoten D[l] (netto +1).
#
#   sum_{e: e[1]=n} f[l,e]  -  sum_{e: e[0]=n} f[l,e]  =
#       -1  falls n = O[l]   (Quelle)
#       +1  falls n = D[l]   (Senke)
#        0  sonst             (Durchgangsknoten)
# ---------------------------------------------------------------------------

# Vorberechnung: ein- und ausgehende Kanten je Knoten
in_edges  = {n: [e for e in E if e[1] == n] for n in N}
out_edges = {n: [e for e in E if e[0] == n] for n in N}

n_c5 = 0
for l in L:
    for n in N:
        inflow  = pl.lpSum(f[(l, e)] for e in in_edges[n])
        outflow = pl.lpSum(f[(l, e)] for e in out_edges[n])

        if n == O_dep[l]:      # Startknoten: Fahrzeug verlässt
            model += (outflow - inflow == 1, f"C5_Flow_Start_{l}_{n}")
        elif n == D_dep[l]:    # Zielknoten: Fahrzeug kommt an
            model += (inflow - outflow == 1, f"C5_Flow_Ziel_{l}_{n}")
        else:                  # Durchgangsknoten: Fluss erhalten
            model += (inflow == outflow,     f"C5_Flow_Trans_{l}_{n}")
        n_c5 += 1

print(f"C5: {n_c5} Constraints hinzugefügt (Flusserhaltung).")

In [ ]:
# ---------------------------------------------------------------------------
# C6: SUBTOUR-ELIMINIERUNG (Miller-Tucker-Zemlin)
# Verhindert, dass der Solver isolierte Kreisläufe (Subtouren) als Lösung
# akzeptiert, die zwar Flusserhaltung erfüllen, aber physikalisch unsinnig sind.
#
#   u[l,i] - u[l,j] + |N| * f[l,(i,j)] <= |N| - 1
#   für alle l in L, (i,j) in E mit j != O[l]
#
# u[l,n] kodiert die Reihenfolge, in der Knoten n auf der Route von l besucht wird.
# ---------------------------------------------------------------------------

n_c6 = 0
M_mtz = len(N)  # Enger M-Wert = Knotenanzahl (mathematisch korrekte Schranke)

for l in L:
    for (i, j) in E:
        if j != O_dep[l]:  # Startknoten ausgenommen (kein Vorgänger)
            model += (
                u[(l, i)] - u[(l, j)] + M_mtz * f[(l, (i, j))] <= M_mtz - 1,
                f"C6_MTZ_{l}_{i}_{j}"
            )
            n_c6 += 1

print(f"C6: {n_c6} Constraints hinzugefügt (MTZ Subtour-Eliminierung).")
print(f"     M_mtz = |N| = {M_mtz}  (engste sinnvolle Schranke — numerisch stabil)")

In [ ]:
# ---------------------------------------------------------------------------
# C7: REICHWEITENBEDINGUNG (E-Truck spezifisch)
# Die Gesamtdistanz einer Route darf die Akkureichweite des zugewiesenen
# Fahrzeugs nicht überschreiten.
#
#   sum_{e in E} Dist[e] * f[l,e]  <=  sum_{v in V} Range[v] * y[l,v]
#   für alle l in L
#
# Hinweis: Ladeinfrastruktur-Zwischenstopps (aus charging_infrastructure_clean.csv)
# könnten die effektive Reichweite erhöhen — hier noch nicht modelliert.
# ---------------------------------------------------------------------------

Range = {v: vehicles_data[v]['range_km'] for v in V}

n_c7 = 0
for l in L:
    total_dist  = pl.lpSum(Dist[e] * f[(l, e)] for e in E)
    max_range   = pl.lpSum(Range[v] * y[(l, v)] for v in V)
    model += (
        total_dist <= max_range,
        f"C7_Reichweite_{l}"
    )
    n_c7 += 1

print(f"C7: {n_c7} Constraints hinzugefügt (Reichweite).")

# Modellzusammenfassung
print()
print("=" * 50)
print("MODELLZUSAMMENFASSUNG")
print("=" * 50)
print(f"Variablen:    {model.numVariables()}")
print(f"Constraints:  {model.numConstraints()}")
print(f"Modelltyp:    {model.sense} ({pl.LpMinimize})")

---
## 7. Modell lösen & Ergebnisse <a id='solve'></a>

Das Modell wird mit dem **CBC-Solver** (Standard in PuLP, Open Source) gelöst.  (Das hier hat die zu 100% KI generiert, müssten das mal genauer anschauen)

In [ ]:
# ---------------------------------------------------------------------------
# MODELL LÖSEN
# ---------------------------------------------------------------------------

solver = pl.PULP_CBC_CMD(
    msg=1,       # Solver-Output anzeigen
    timeLimit=60 # Zeitlimit in Sekunden
)

status = model.solve(solver)

print()
print("=" * 50)
print(f"Status:         {pl.LpStatus[status]}")
print(f"Zielfunktion:   {pl.value(model.objective):.6f}")
print("=" * 50)

In [ ]:
# ---------------------------------------------------------------------------
# ERGEBNISAUSWERTUNG
# ---------------------------------------------------------------------------

if pl.LpStatus[status] in ['Optimal', 'Feasible']:

    print("\n--- Fahrzeugzuweisungen (y[l,v] = 1) ---")
    assignments = []
    for l in L:
        for v in V:
            if pl.value(y[(l, v)]) is not None and pl.value(y[(l, v)]) > 0.5:
                assignments.append({
                    'Lieferung': l,
                    'Fahrzeug':  v,
                    'Gewicht_t': Dem[l],
                    'Klasse':    Class[l],
                    'Ziel':      D_dep[l]
                })
    if assignments:
        print(pd.DataFrame(assignments).to_string(index=False))
    else:
        print("  Keine Zuweisungen gefunden.")

    print("\n--- Aktive Fahrzeuge (z[v] = 1) ---")
    for v in V:
        val = pl.value(z[v])
        if val is not None and val > 0.5:
            print(f"  {v}  (Fixkosten: {FC[v]} €/Tag)")

    print("\n--- Genutzte Kanten (f[l,e] = 1) ---")
    routes = {l: [] for l in L}
    for l in L:
        for e in E:
            val = pl.value(f[(l, e)])
            if val is not None and val > 0.5:
                routes[l].append(e)
        if routes[l]:
            print(f"  {l}: {' → '.join([r[0] for r in routes[l]] + [routes[l][-1][1]])}")
        else:
            print(f"  {l}: keine Route (prüfe Netzwerk-Konnektivität)")

    print("\n--- Kostenaufschlüsselung ---")
    fix_cost_val = sum(FC[v] * pl.value(z[v]) for v in V
                       if pl.value(z[v]) is not None)
    print(f"  Fixkosten:      {fix_cost_val:.2f} €")
    print(f"  Zielfunktion:   {pl.value(model.objective):.6f}  (normiert)")

else:
    print(f"Kein optimales Ergebnis gefunden. Status: {pl.LpStatus[status]}")
    print("Mögliche Ursachen:")
    print("  - Kein zulässiger Pfad vom Depot zum Ziel (Netzwerk nicht verbunden)")
    print("  - Gefahrgutrestriktionen sperren alle Routen")
    print("  - Kapazität der Fahrzeuge zu gering für die Lieferungen")